# Full ColorFeret cGAN Training
This notebook orchestrates end-to-end training on the complete 11,338 image ColorFeret dataset.
It uses the pre-uploaded Kaggle dataset and linked pretrained weights securely to train the GitHub repository on Kaggle.

## 1. Check Kaggle GPU & Setup Environment

In [ ]:
import os
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the GitHub Repository

In [ ]:
!rm -rf stylized-portrait-generation
!git clone https://github.com/supratimcoder1/stylized-portrait-generation.git
%cd stylized-portrait-generation

## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt -q

## 4. Verify Dataset & Weights Location

In [ ]:
# Assumes Kaggle Dataset names: color-feret-complete and best-model-pth
images_dir = "/kaggle/input/color-feret-complete/images"
weights_path = "/kaggle/input/best-model-pth/best_model.pth"

print(f"Images dir exists: {os.path.exists(images_dir)}")
if os.path.exists(images_dir):
    print(f"Total images: {len(os.listdir(images_dir))} files")
print(f"Pretrained weights exist: {os.path.exists(weights_path)}")

## 5. Create Dataset Symlink and Model Setup
Link the read-only Kaggle inputs into the repo's expected local mapping.

In [ ]:
!rm -rf dataset
!mkdir -p dataset/images
!mkdir -p checkpoints

# Symlink the ColorFeret images and pretrained weights
!ln -s {images_dir}/* dataset/images/
!ln -s {weights_path} checkpoints/best_model.pth

print("Linked images snippet:")
!ls -la dataset/images/ | head -n 5
print("\nLinked checkpoints layout:")
!ls -la checkpoints/

## 6. Start Training

In [ ]:
!python training/train.py --epochs 100 --batch-size 8 --output-dir /kaggle/working/training_outputs --checkpoint-dir ./checkpoints

## 7. Inspect Training Outputs

In [ ]:
import glob
from IPython.display import display, Image as IPImage

sample_files = sorted(glob.glob("/kaggle/working/training_outputs/epoch_*.png"))
if sample_files:
    latest = sample_files[-1]
    print(f"Showing: {latest}")
    display(IPImage(filename=latest, width=800))
else:
    print("No sample images found - training may not have completed.")

## 8. Download Updated Best Model

In [ ]:
import shutil
# Check and copy the updated GAN path to the working directory for download
if os.path.exists("./checkpoints/best_model_cgan.pth"):
    shutil.copy("./checkpoints/best_model_cgan.pth", "/kaggle/working/best_model_cgan.pth")
    print("Model ready in /kaggle/working/best_model_cgan.pth")

!zip -j /kaggle/working/final_model.zip /kaggle/working/best_model_cgan.pth
print("Updated cGAN best model zipped to /kaggle/working/final_model.zip")